In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv
import cohere

In [2]:
import openai
load_dotenv()

True

In [3]:
client=OpenAI()

In [4]:
QDRANT_URL = os.getenv("QDRANT_URL")

In [4]:
QDRANT_URL

'http://localhost:6335'

In [9]:
qdrant_client=QdrantClient(url=QDRANT_URL)

In [6]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    vectors_config={
        "text-embedding-model-3-small":VectorParams(size=1536,distance=Distance.COSINE)
    }
    ,
    sparse_vectors_config={
        "bm25":SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [7]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

In [8]:
def get_embedding_batch(text_list,model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response=openai.embeddings.create(input=text_list,model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings=[]
    counter=1
    for i in range(0,len(text_list),batch_size):
        batch=text_list[i:i+batch_size]
        response=openai.embeddings.create(input=batch,model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        counter+=1
    
    return all_embeddings

In [9]:
df_items=pd.read_json("C:/Users/Anurag/OneDrive/Desktop/coading/Ai-engineering/Data/meta_Electronics_2022_23_with_categeory_rating_100_sample_2000.jsonl",lines=True)

In [10]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,Gvoears Replacement Ear Pads Cushions for Skul...,4.6,129,[【Perfect Compatibility】These ear pads are mad...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'the installation of earpads for sk...,G GVOEARS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '3.94 x 3.15 x 0.98 inc...,B0BTBHYWY4,NaN,NaN,NaN
1,All Electronics,"Ethernet Splitter,NOBVEQ 1 Male to 2 Female Ne...",4.5,4654,[NOBVEQ RJ45 Ethernet Splitter: It converts th...,[],10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NOBVEQ,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '7.68 x 4.09 x 1.06 inc...,B0BWXS9QVY,NaN,NaN,NaN
2,Cell Phones & Accessories,Case for iPad 9th Generation 10.2 Inch 2021 iP...,4.5,110,[【Compatibility】 - Designed for 10.2-Inch iPad...,[],33.90,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Case for iPad 9th/8th/7th generati...,H HOLIMET,"[Electronics, Computers & Accessories, Tablet ...",{'Package Dimensions': '10.16 x 7.87 x 0.75 in...,B0BYCT8F2V,NaN,NaN,NaN
3,Camera & Photo,"Aipmoz Hidden Camera with 32G SD Card, Spy Cam...",4.8,107,[【Bluetooth Speaker Hidden Camera】This hidden ...,[],89.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'How to connect this bluetooth spea...,Aipmoz,"[Electronics, Camera & Photo, Video Surveillan...","{'Product Dimensions': '3.3 x 3.3 x 4 inches',...",B0CB1XQ9NR,NaN,NaN,NaN
4,All Electronics,"Military Smart Watch Men(Answer/Make Calls), 2...",4.0,136,[CUSTOMIZED FOR THE ADVENTURER: Our military s...,[],35.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'A tactical smartwatch for your out...,Moowhsh,"[Electronics, Wearable Technology, Smartwatches]",{'Product Dimensions': '9.84 x 8.66 x 0.79 inc...,B0BCF3KRMM,NaN,NaN,NaN


In [11]:
def preprocessed_decription(row):
    return f"{row['title']}{''.join(row['features'])}"

In [12]:
def extrect_first_large_image(row):
    return row['images'][0].get("large","")

In [13]:
df_items['description']=df_items.apply(preprocessed_decription,axis=1)
df_items['image']=df_items.apply(extrect_first_large_image,axis=1)

In [14]:
df_items

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,All Electronics,Gvoears Replacement Ear Pads Cushions for Skul...,4.6,129,[【Perfect Compatibility】These ear pads are mad...,Gvoears Replacement Ear Pads Cushions for Skul...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'the installation of earpads for sk...,G GVOEARS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '3.94 x 3.15 x 0.98 inc...,B0BTBHYWY4,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41SY-3EDUy...
1,All Electronics,"Ethernet Splitter,NOBVEQ 1 Male to 2 Female Ne...",4.5,4654,[NOBVEQ RJ45 Ethernet Splitter: It converts th...,"Ethernet Splitter,NOBVEQ 1 Male to 2 Female Ne...",10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NOBVEQ,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '7.68 x 4.09 x 1.06 inc...,B0BWXS9QVY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31neVw1ZCt...
2,Cell Phones & Accessories,Case for iPad 9th Generation 10.2 Inch 2021 iP...,4.5,110,[【Compatibility】 - Designed for 10.2-Inch iPad...,Case for iPad 9th Generation 10.2 Inch 2021 iP...,33.90,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Case for iPad 9th/8th/7th generati...,H HOLIMET,"[Electronics, Computers & Accessories, Tablet ...",{'Package Dimensions': '10.16 x 7.87 x 0.75 in...,B0BYCT8F2V,NaN,NaN,NaN,https://m.media-amazon.com/images/I/516BZ2RJmS...
3,Camera & Photo,"Aipmoz Hidden Camera with 32G SD Card, Spy Cam...",4.8,107,[【Bluetooth Speaker Hidden Camera】This hidden ...,"Aipmoz Hidden Camera with 32G SD Card, Spy Cam...",89.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'How to connect this bluetooth spea...,Aipmoz,"[Electronics, Camera & Photo, Video Surveillan...","{'Product Dimensions': '3.3 x 3.3 x 4 inches',...",B0CB1XQ9NR,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41Qvn7tuwB...
4,All Electronics,"Military Smart Watch Men(Answer/Make Calls), 2...",4.0,136,[CUSTOMIZED FOR THE ADVENTURER: Our military s...,"Military Smart Watch Men(Answer/Make Calls), 2...",35.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'A tactical smartwatch for your out...,Moowhsh,"[Electronics, Wearable Technology, Smartwatches]",{'Product Dimensions': '9.84 x 8.66 x 0.79 inc...,B0BCF3KRMM,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41nCIVgk2u...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Cell Phones & Accessories,CYNOVA Insta 360 X3 Sticky Lens Guard Screen T...,3.9,188,[Insta360 x3 Lens Guard : Protection for both ...,CYNOVA Insta 360 X3 Sticky Lens Guard Screen T...,23.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Insta360 X3 Sticky Lens Guards', '...",CYNOVA,"[Electronics, Camera & Photo, Accessories, Dig...","{'Product Dimensions': '1""L x 1""W', 'Item Weig...",B0BJVCLHHW,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41yUJULN8L...
1996,Amazon Home,BulbaCraft 100Pcs Children Dental Stickers for...,4.5,174,[🦷Enjoy the High-Quality Vinyl Material - dent...,BulbaCraft 100Pcs Children Dental Stickers for...,8.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],BulbaCraft,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'BulbaCraft', 'Room Type': 'Bathroom...",B0BPJJBNY2,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51sTRVI11w...
1997,Camera & Photo,"Rbcior Pet Camera, Indoor Camera Surveillance ...",4.3,106,[],"Rbcior Pet Camera, Indoor Camera Surveillance ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Rbcior,[],{'Package Dimensions': '6.06 x 3.82 x 3.7 inch...,B09TP8231P,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41cSGzmsZi...
1998,Home Audio & Theater,"Phinistec 80GB HiFi MP3 Player with Bluetooth,...",3.8,101,[♪-【High-Fidelity Stereo Sound & DSD Codec】: T...,"Phinistec 80GB HiFi MP3 Player with Bluetooth,...",NaN,[{'thumb': 'https://m.media-amazon.com

In [15]:
data_to_embed=df_items[["description","image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [16]:
data_to_embed[0]['description']

'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, this headphone ear covers set insulates yo

In [17]:
text_to_embed=[data["description"] for data in data_to_embed]

In [18]:
embeddings = get_embedding_batch(text_to_embed)


In [21]:
len(embeddings[0])

1536

In [22]:
pointstructs=[]
i=1
for embedding,data in zip(embeddings,data_to_embed):
    if i==1:print(embedding,data)
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-model-3-small":embedding,
                "bm25":Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i+=1

[0.014129638671875, 0.0020885467529296875, 0.01305389404296875, 0.00627899169921875, -0.0390625, -0.039306640625, 0.0487060546875, 0.01971435546875, 0.026763916015625, -0.0552978515625, 0.0190582275390625, -0.01300811767578125, -0.0391845703125, 0.01412200927734375, -0.0023441314697265625, 0.0538330078125, 0.040985107421875, 0.006565093994140625, -0.01160430908203125, 0.008270263671875, -0.005741119384765625, 0.066650390625, 0.05224609375, 0.0103759765625, 0.041717529296875, 0.040740966796875, 0.01378631591796875, 0.0204010009765625, 0.054107666015625, 0.003719329833984375, -0.047210693359375, -0.0025482177734375, -0.006290435791015625, -0.05059814453125, -0.05450439453125, -0.003330230712890625, -0.02001953125, -0.0178985595703125, 0.00611114501953125, 0.008636474609375, 0.050689697265625, 0.01422882080078125, 0.0039520263671875, -0.0298919677734375, -0.0264129638671875, 0.026611328125, -0.031646728515625, 0.028839111328125, 0.001232147216796875, 0.0377197265625, 0.0103759765625, 0.03

In [23]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [24]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[500:1000],
    wait=True
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1000:1500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [26]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1500:2000],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [27]:
client=OpenAI()

In [7]:
def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [5]:
def reteriver_data(query, quadrant_client, k):
    query_embedding = get_embedding(query)

    result = quadrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-model-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )

        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )

In [10]:
result=reteriver_data("can i get some earphones",qdrant_client,k=10)

In [15]:
result

(['Wireless EarbudsWireless Earbuds.Wireless Earbuds.Wireless Earbuds.Wireless Earbuds.Wireless Earbuds.',
  'Cleaner Kit for AirPod Pro 1 2 3, AMEAMI Earbud Cleaning Pen Tool for iPhone Samsung Lego Huawei MI Wireless Headphone (White)🔴【SEPARATED DUAL HEAD DESIGN】3-in-1 airpods cleaning kit with Flocking Sponge, High-density Brush and Metal Pen Tip, meet a variety of cleaning airpods, earbuds, headphone, earphone, case needs🔴【HIGH-DENSITY BRUSH & FLOCKING SPONGE】There are a soft microfiber brush and a lightweight flocking sponge above the cleaning pen, which can deeply clean the small parts, holes and charging bin of the AirPods. It is beneficial to improve headphones sound quality, charging efficiency and prolong service life.🔴【METAL CLEANER PEN TIP】Metal cleaning pen tip can clean the stubborn dust, thoroughly clean the gap.🔴【PERFECT COMBINATION】You will get a 3-in-1 airpod cleaning kit and 6 pieces of wet & dry wipe. It will keep your airpods or wireless headphones as clean as poss

In [11]:
CO_API_KEY=os.getenv("CO_API_KEY")

In [12]:

cohere_client=cohere.ClientV2(CO_API_KEY)

In [17]:
response_rerank=cohere_client.rerank(
    model= "rerank-v4.0-fast",##"rerank-v4.0-pro",
    query="can i get some earphones",
    documents=result[0],
    top_n=10
)


In [23]:
response_rerank.results[0].index

6

In [24]:
reranked_products = []

for item in response_rerank.results:
    idx = item.index

    reranked_products.append({
        "product_id": result[1][idx],
        "rating": result[2][idx],
        "fusion_score": result[3][idx],
        "image": result[4][idx],
        "price": result[5][idx],
        "description": result[0][idx],
        "rerank_score": item.relevance_score,
    })

In [25]:
reranked_products

[{'product_id': 'B087CV5K43',
  'rating': 4.4,
  'fusion_score': 0.21666667,
  'image': 'https://m.media-amazon.com/images/I/414fjefXE7L._AC_.jpg',
  'price': 23.99,
  'description': 'YINYOO KBEAR Rosefinch in Ear Earphones Headphones, Wired Earbuds with Microphone with Clear Sound Detachable Cable 3.5mm Plug for Computer, Smartphone, PC, Calling, Home (Black, with mic)▶ 【Good sound quality】 Based on the Harman Curve, the tuning of Rosefinch has enhanced the atmosphere and quantity of bass to present a more dynamic and thicker sound signature.The Unit owns low impedance and high sensitivity, Mobile phones can also be easily pushed directly. Bring you a new kind of sense.▶ 【3.5mm plug, compatible cable】 Professional 4 core OFC wire soft and durable. The 3.5mm plug can work with all 3.5mm related devices, Androids, smartphones,pc, phones, MP3/MP4 players, music speakers.▶ 【Detachable 2pins 0.78mm Connector】The detachable design improves the playability, lets you can exchange other HiFi c